# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. `20 10`. `s["b"]` берёт значение по ярлыку "b", а `s.iloc[0]` по позиции, это первый элемент 10.
2. `Алматы NaN`, `Астана NaN`, `Шымкент 3.0`. Pandas складывает по ярлыкам: общий ярлык только Шымкент (2 + 1 = 3), у остальных пары нет, поэтому NaN. Из-за NaN тип становится float, отсюда 3.0, а ярлыки в результате идут по алфавиту.
3. `(2, 2)`. Условию x > 1 подходят две строки, столбцов по-прежнему два.
4. `Series DataFrame`. Одни скобки дают столбец как Series, а список имён в двойных скобках даёт таблицу, даже если имя одно.
5. `[1, 2] 5`. После фильтра у строк остаются старые ярлыки 1 и 2. `iloc[0, 1]` берёт первую строку по порядку и второй столбец `y`, там 5.

In [1]:
# проверка предсказаний
import pandas as pd

s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])

20 10
Алматы     NaN
Астана     NaN
Шымкент    3.0
dtype: float64
(2, 2)
Series DataFrame
[1, 2] 5


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [4]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [5]:
# 1. паспорт таблицы
print("строк и столбцов:", df.shape)
print(df.dtypes)

n_days = df["delivery_days"].count()   # count не считает пропуски
print("\nпропусков в delivery_days:", len(df) - n_days)

for col in ["city", "category", "status", "customer_id", "order_id"]:
    print(f"разных значений в {col}: {df[col].nunique()}")

print("\nнаписания города:", df["city"].unique().tolist())
print("заказов TEST:", (df["customer_id"] == "TEST").sum(),
      "| сумма:", df.loc[df["customer_id"] == "TEST", "amount"].sum())

строк и столбцов: (5463, 8)
order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object

пропусков в delivery_days: 160
разных значений в city: 14
разных значений в category: 5
разных значений в status: 2
разных значений в customer_id: 2003
разных значений в order_id: 5283

написания города: ['Алматы', 'Астана', 'Караганда', 'Шымкент', 'алматы', 'Атырау', 'шымкент', 'Shymkent', 'астана', 'Актобе', 'Astana ', 'АЛМАТЫ', ' Алматы', 'Almaty']
заказов TEST: 3 | сумма: 29999970.0


**Ответ:** в таблице 5 463 строки и 8 столбцов. Числа только в `order_id`, `amount` (после готовой строки) и `delivery_days`, остальное текст. В `delivery_days` 160 пропусков. Разных значений: 14 в `city`, 5 в `category`, 2 в `status`, 2 003 в `customer_id`.

**Менеджеру, три проблемы данных:**

1. Городов 6, а записаны они 14 способами: Алматы, например, встречается как `Алматы`, `алматы`, `АЛМАТЫ`, ` Алматы` и `Almaty`. Если считать выручку по одному написанию, часть заказов города просто потеряется.
2. Строк 5 463, а разных `order_id` только 5 283, то есть 180 строк повторяются. Каждый повтор ещё раз добавляет свою сумму в выручку.
3. Есть тестовый клиент TEST: три заказа по 9 999 990 ₸, почти 30 млн ₸ ненастоящей выручки. Он сильно завышает и выручку, и средний чек.

И ещё одна мелочь, которая тоже мешает: суммы в файле записаны текстом (`"15 990"`), и без перевода в числа `sum()` их не складывает, а склеивает.

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [6]:
# 2. типичный заказ: оплаченные без TEST
x = df.loc[paid & not_test, "amount"]
print("заказов:", len(x))
print(f"среднее:  {x.mean():,.0f} ₸")
print(f"медиана:  {x.median():,.0f} ₸")
print(f"25%:      {x.quantile(0.25):,.0f} ₸")
print(f"75%:      {x.quantile(0.75):,.0f} ₸")
print(f"среднее выше медианы на {(x.mean() / x.median() - 1) * 100:.0f}%")

заказов: 4901
среднее:  14,672 ₸
медиана:  10,300 ₸
25%:      5,880 ₸
75%:      18,690 ₸
среднее выше медианы на 42%


**Ответ:** по 4 901 оплаченному заказу без TEST среднее 14 672 ₸, медиана 10 300 ₸, а половина всех заказов лежит между 5 880 и 18 690 ₸. Типичный чек около 10 тысяч тенге.

В отчёте честнее показывать медиану. Среднее выше неё на 42%, его тянут вверх немногие дорогие заказы, в основном электроника. На медиану они почти не влияют, поэтому она лучше показывает обычную покупку.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [7]:
# 3. доля отмен: все заказы, Шымкент и Алматы (все написания)
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
cancel = df["status"] == "отменён"

all_share = cancel.mean()
sh_share = cancel[df["city"].isin(SH)].mean()
al_share = cancel[df["city"].isin(AL)].mean()

print(f"все заказы: {all_share:.1%}")
print(f"Шымкент:    {sh_share:.1%}  (заказов {df['city'].isin(SH).sum()})")
print(f"Алматы:     {al_share:.1%}  (заказов {df['city'].isin(AL).sum()})")
print(f"Шымкент / Алматы: {sh_share / al_share:.1f} раза, Шымкент / все: {sh_share / all_share:.1f} раза")

все заказы: 10.2%
Шымкент:    20.4%  (заказов 804)
Алматы:     7.9%  (заказов 2188)
Шымкент / Алматы: 2.6 раза, Шымкент / все: 2.0 раза


**Ответ:** в среднем отменяют 10,2% заказов. В Шымкенте 20,4%, в Алматы 7,9%. Шымкент выделяется: там отменяют в 2,6 раза чаще, чем в Алматы, и в 2 раза чаще, чем в среднем по всем городам.

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [8]:
# 4. выручка оплаченных без TEST по месяцам, дубли пока не убираем
for name, cities in [("Шымкент", SH), ("Алматы", AL)]:
    m = paid & not_test & df["city"].isin(cities)
    rev_sep = df.loc[m & sep, "amount"].sum()
    rev_oct = df.loc[m & octo, "amount"].sum()
    print(f"{name:<8} сентябрь {rev_sep:>12,.0f} ₸  октябрь {rev_oct:>12,.0f} ₸  "
          f"изменение {(rev_oct / rev_sep - 1) * 100:.1f}%")

Шымкент  сентябрь    4,949,800 ₸  октябрь    4,044,010 ₸  изменение -18.3%
Алматы   сентябрь   15,523,940 ₸  октябрь   14,643,020 ₸  изменение -5.7%


**Ответ:** Шымкент 4 949 800 → 4 044 010 ₸, это -18,3%. Алматы 15 523 940 → 14 643 020 ₸, -5,7%. В Шымкенте выручка упала в 3,2 раза сильнее.

**Менеджеру:** проблема в первую очередь в Шымкенте. Падение Алматы здесь, скорее всего, преувеличено: все 180 повторяющихся строк приходятся на сентябрь (это видно ещё в ДЗ 2), а по чистым данным в ДЗ 4 Алматы в октябре даже выросла примерно на 1%.

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [9]:
# 5. ошибки в коде из условия
try:
    cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
except TypeError as e:
    print("падает:", e)

# ошибка 1: & выполняется раньше ==, поэтому каждое условие нужно взять в скобки
only_one = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]
print("со скобками, но одно написание:", len(only_one))

# ошибка 2: "шымкент" и "Shymkent" не попадают, нужен isin со всеми написаниями
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print("правильно:", len(cancel_sh))

падает: unsupported operand type(s) for &: 'str' and 'StringArray'
со скобками, но одно написание: 153
правильно: 164


**Ответ:** первая ошибка роняет код: без скобок `&` выполняется раньше `==`, получается `"Шымкент" & df["status"]`, и Python падает с TypeError. Вторая ошибка тихая: город сравнивается только с написанием `Шымкент`, поэтому `шымкент` и `Shymkent` теряются. С одними скобками код выдал бы 153, а на самом деле отменённых заказов в Шымкенте 164.

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [10]:
# 6. самые частые клиенты
top5 = df["customer_id"].value_counts().head(5)
print(top5)

top = top5.index[0]
cols = ["order_id", "order_date", "city", "amount", "status"]
orders_top = df.loc[df["customer_id"] == top, cols]
print(f"\nзаказы {top}:")
print(orders_top)

ids = orders_top["order_id"].value_counts()
print(f"\nстрок {len(orders_top)}, разных order_id {orders_top['order_id'].nunique()}")
print("повторяются:", ids[ids > 1].to_dict())
print(f"\nпо всей таблице: строк {len(df)}, разных order_id {df['order_id'].nunique()}")

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
Name: count, dtype: int64

заказы C01818:
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен

строк 11, разных order_id 10
повторяются: {101504: 2}

по всей таблице: строк 5463, разных order_id 5283


**Ответ:** больше всех заказов у клиента C01818, 11 штук. Но разных `order_id` у него только 10: заказ 101504 от 15.09 на 8 890 ₸ записан два раза.

Значит, в выручке, которую мы считали весь урок, такие повторы посчитаны дважды. По всей таблице это 180 лишних строк (5 463 строки на 5 283 заказа), поэтому до любого расчёта выручки дубли надо убирать.

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [11]:
# 7. выбросы по правилу IQR: оплаченные без TEST
q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
out = x > upper

print(f"Q1 {q1:,.0f}, Q3 {q3:,.0f}, IQR {iqr:,.0f}, граница {upper:,.0f} ₸")
print(f"выше границы: {out.sum()} заказов, доля {out.mean():.1%}")

# категории выбросов: берём строки df по ярлыкам из x
out_cat = df.loc[x[out].index, "category"].value_counts()
print(out_cat)
print("чаще всего:", out_cat.index[0])

Q1 5,880, Q3 18,690, IQR 12,810, граница 37,905 ₸
выше границы: 320 заказов, доля 6.5%
category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64
чаще всего: Электроника


**Ответ:** Q1 = 5 880 ₸, Q3 = 18 690 ₸, IQR = 12 810 ₸, граница 37 905 ₸. Выше неё 320 заказов (6,5%), и почти все это Электроника: 308 из 320.

По z-оценке в ДЗ 4 выбросов было 87, а граница там получалась около 54 600 ₸. Правила дают разное число, потому что z-оценка опирается на среднее и стандартное отклонение, а их раздувают те же дорогие заказы, и граница уезжает выше. IQR строится по квартилям, на которые дорогие заказы почти не влияют, поэтому при суммах с длинным хвостом вправо он отмечает больше заказов. К тому же здесь дубли ещё не убраны, а в ДЗ 4 считали по чистым данным.

**Менеджеру:** это не ошибки в данных, а обычная дорогая электроника. Удалять такие заказы нельзя, но и считать по ним типичный чек тоже не стоит.

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [12]:
# 8. доля отмен для каждого написания города
cancel_by_city = df.loc[df["status"] == "отменён", "city"].value_counts()
all_by_city = df["city"].value_counts()

# Series стоят в разном порядке, но pandas делит по ярлыкам
print("порядок в отменах:", cancel_by_city.index[:4].tolist())
print("порядок во всех:  ", all_by_city.index[:4].tolist())

cancel_share = (cancel_by_city / all_by_city).sort_values(ascending=False)
print(cancel_share.round(3))

print("\nпроверка по ярлыку Шымкент:", round(cancel_by_city["Шымкент"] / all_by_city["Шымкент"], 3))
for c in SH:
    print(f"{c:<9} заказов {all_by_city[c]:>4}, отменено {cancel_by_city[c]:>3}, доля {cancel_share[c]:.1%}")

порядок в отменах: ['Алматы', 'Шымкент', 'Астана', 'Караганда']
порядок во всех:   ['Алматы', 'Астана', 'Шымкент', 'Караганда']
city
Shymkent     0.231
Шымкент      0.215
алматы       0.141
астана       0.131
Актобе       0.103
Astana       0.100
Караганда    0.089
Астана       0.085
Алматы       0.080
Атырау       0.075
Almaty       0.058
 Алматы      0.045
шымкент      0.037
АЛМАТЫ       0.032
Name: count, dtype: float64

проверка по ярлыку Шымкент: 0.215
Шымкент   заказов  711, отменено 153, доля 21.5%
шымкент   заказов   54, отменено   2, доля 3.7%
Shymkent  заказов   39, отменено   9, доля 23.1%


**Ответ:** деление идёт по ярлыкам: в двух Series города стоят в разном порядке, а доля для ярлыка `Шымкент` совпадает с ручным расчётом 153 / 711. У трёх написаний Шымкента доли совсем разные: `Shymkent` 23,1% (39 заказов), `Шымкент` 21,5% (711), `шымкент` 3,7% (54).

По одному написанию судить о городе нельзя. Город при этом разбит на куски, а в маленьких группах на 39 или 54 заказа пара отмен сильно двигает долю. Вместе три написания дают 164 отмены из 804 заказов, те же 20,4%. Поэтому перед анализом названия городов надо привести к одному виду: убрать лишние пробелы, сделать один регистр и заменить латиницу на кириллицу.